# Comprehensive TikTok Data Collection for Computational Social Science

**IC2S2 2026 Tutorial**
Gayoung Jeon, Cameron Moy, Deen Freelon
Annenberg School for Communication, University of Pennsylvania

---

This notebook is the hands-on companion to our IC2S2 2026 tutorial. It introduces TikTok data collection through three different tools — the official **TikTok Research API**, **Apify** (a commercial cloud scraping platform), and **Pyktok** (an open-source browser-based scraper). The three tools differ substantially in cost, access requirements, and the data they return; our stress-testing research finds they frequently produce strikingly different results even when given identical queries.

**Pyktok** is an open-source library maintained by Gayoung Jeon and Deen Freelon (will be released soon). This tutorial demonstrates multiple endpoints including User, Hashtag, Keyword, and Sound. The code is provided **as-is** and you use it at your own risk.

You do not need to use all three tools. Each section is self-contained — follow whichever apply to your setup.

**Sections:**
- [1. Setup](#1.-Setup)
- [2-1. TikTok Research API](#2-1.-TikTok-Research-API) — User, Keyword, Hashtag, Comments
- [2-2. Apify](#2-2.-Apify) — User, Keyword, Hashtag, Comments, Related Videos
- [2-3. Pyktok](#2-3.-Pyktok) — User, Hashtag, Keyword, Sound

## Tools at a Glance

| | Research API | Apify | Pyktok |
|---|---|---|---|
| **Access** | Academic application (takes weeks) | Free tier (~$5 credit) | Free, open source |
| **Cost** | Free | ~$0.25 - $1.00 per 1,000 results | Free |
| **Endpoints** | User, Keyword, Hashtag, Comments | User, Keyword, Hashtag, Comments, Related | User, Hashtag, Keyword, Sound, Comments, Related |
| **Data source** | Back-end API | Front-end cloud scrape | Front-end browser scrape |
| **Daily limits** | 1,000 API calls/day | Credit-based | TikTok rate limits apply |
| **Auth required** | Yes (OAuth token) | API token | Optional (sessionid cookie) |

---
## 1. Setup

Install everything before collecting. Run these cells once at the start
of each session.


In [ ]:
# TikTok Research API wrapper + utilities
!pip install -q TikTokResearchApi python-dotenv pandas requests tqdm

In [ ]:
# Apify client (used in section 2-2)
!pip install -q apify-client

In [ ]:
# pykto
# Installs the local copy in editable mode + Playwright's Chromium binary
!pip install -q -e .
!playwright install chromium

### 1.1 Credentials

Create a file called `credential.env` in the same directory as this notebook
(a `credential.env.example` is included in the repo as a template):

```
# TikTok Research API
CLIENT_KEY=your_client_key_here
CLIENT_SECRET=your_client_secret_here

# Apify
APIFY_API_TOKEN=your_apify_token_here
```

Where to find these:
- **TikTok API credentials:** [developers.tiktok.com/products/research-api](https://developers.tiktok.com/products/research-api/). Apply at least a month before your project; review takes time.
- **Apify token:** apify.com → Settings → Integrations → API Token.
- **Pyktok:** Optional sessionid cookie for login-required endpoints. See Section 2-3 for details.

> Do not commit `credential.env`. It is already listed in `.gitignore`.

In [ ]:
# Verify your credentials loaded correctly
import os
from dotenv import load_dotenv

load_dotenv("credential.env", override=True)

api_key   = os.getenv("CLIENT_KEY")
apify_tok = os.getenv("APIFY_API_TOKEN")

print("Research API credentials:", "loaded" if api_key   else "MISSING - check credential.env (only needed for section 2-1)")
print("Apify token:             ", "loaded" if apify_tok else "MISSING - check credential.env (only needed for section 2-2)")
print("Pyktok:                   no credentials needed for public endpoints (section 2-3)")

---
## 2-1. TikTok Research API

The Research API is TikTok's official data access program for academic researchers. Because it queries the back-end directly, it bypasses the recommendation algorithm—you're not getting "what TikTok would show a user," you're asking the database directly. That's valuable for some research questions and less useful for others.

You need an approved developer account to use it. Once approved, TikTok gives you a `client_key` and `client_secret`, which you exchange for a short-lived OAuth token. Tokens expire every **2 hours**, so we'll set up automatic renewal.

### API Quota Limits

TikTok enforces a **hard limit of 1,000 API calls per day**, shared across all endpoints. Here's what that means in practice:

| Endpoint | Supported Queries | Max Results per Call | Daily Call Budget | Max Results/Day |
|---|---|---|---|---|
| `/video/query/` | Username | 100 videos | shared 1,000 calls | ~100,000 |
| `/video/query/` | Keyword | 100 videos | shared 1,000 calls | ~100,000 |
| `/video/query/` | Hashtag | 100 videos | shared 1,000 calls | ~100,000 |
| `/comment/list/` | Video ID | 100 comments | shared 1,000 calls | ~100,000 |

**Things to watch for:**
- The 1,000 calls/day is a **shared pool** across all endpoints. Spend 500 on keywords, you only have 500 left for everything else that day.
- The API tends to surface recent and popular content. Running the same query twice may not return the same videos.
- Some queries return fewer results than the quota technically allows — this is an undocumented behavior we encountered repeatedly in stress testing.
- Always validate that returned videos are still accessible on TikTok at the time of collection.

### Getting Started: Simple Setup

For the tutorial, we keep this short. Pass your credentials to `TikTokResearchAPI` and you get back a client that handles authentication internally. That's it — no token management needed for a session under two hours.

In [ ]:
import os
import pandas as pd
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv
from tiktok_research_api import *

load_dotenv("credential.env", override=True)

# Create the API client — the library fetches and caches the OAuth token internally
api = TikTokResearchAPI(
    client_key=os.getenv("CLIENT_KEY"),
    client_secret=os.getenv("CLIENT_SECRET"),
    qps=2,  # 2 requests per second — conservative to avoid rate limiting
)
print("API client ready")

### 2-1a. User Videos

The user endpoint returns videos posted by a specific account. One call returns up to 100 videos; for most academic use cases (collecting ~50 recent posts per user), you'll use just one call per account.

In [ ]:
# CHANGE VIDEO_FIELDS TO ADD OR REMOVE FIELDS YOU WANT FROM THE API
# Reference: https://developers.tiktok.com/doc/research-api-specs-query-videos

VIDEO_FIELDS = [
    "id", "username", "region_code", "favorites_count", #USER identifiers
    "create_time", "video_description", "voice_to_text", "video_duration", "video_mention_list", "video_label", "video_tag",# Video text and metadata
    "hashtag_names", "hashtag_info_list", # Hashtags
    "share_count", "view_count", "like_count", "comment_count", # Engagement metrics
    "music_id","effect_ids","playlist_id","is_stem_verified", # Audio, effects, and playlist info
    "sticker_info_list", "effect_info_list" # Stickers and effects
]

| Query Feature | What It Means | Example |
|---|---|---|
| `and_criteria` | All listed conditions must be true. | Videos must match both a hashtag and a region. |
| `or_criteria` | At least one listed condition must be true. | Videos can match either one hashtag or another hashtag. |
| `not_criteria` | Listed conditions must not be true. | Exclude videos from a specific region or hashtag. |

| Operation | Meaning | Example Use |
|---|---|---|
| `EQ` | Equal to one value. | `hashtag_name` equals `"hashtag"` |
| `IN` | Matches any value in a list. | `region_code` is in `["US", "CA"]` |
| `GT` | Greater than. | `create_time` is after a timestamp. |
| `GTE` | Greater than or equal to. | `create_time` is at or after a timestamp. |
| `LT` | Less than. | `create_time` is before a timestamp. |
| `LTE` | Less than or equal to. | `create_time` is at or before a timestamp. |

Example query:

```python
query_criteria_1 = Criteria(
    operation="EQ",
    field_name="hashtag_name",
    field_values=["hashtag"],
)

query_criteria_2 = Criteria(
    operation="IN",
    field_name="region_code",
    field_values=["region"],
)

query = Query(and_criteria=[query_criteria_1, query_criteria_2])

In [ ]:
# Set your target
# CHANGE USERNAME TO YOUR OWN TARGET TIKTOK ACCOUNT (no '@')
username   = "drjessicaknurick"

# CHANGE START_DATE TO YOUR EARLIEST DATE (YYYYMMDD) - both start_date and end_date are REQUIRED by the API
start_date = "20220106" #oldest video of the target account https://www.tiktok.com/@drjessicaknurick
end_date   = datetime.now(timezone.utc).strftime("%Y%m%d")  # up to today

# Build the query

request = QueryVideoRequest(
    fields=",".join(VIDEO_FIELDS),
    query=Query(and_criteria=[Criteria("EQ", "username", [username])]),
    start_date=start_date,
    end_date=end_date,
    max_count=100,  # videos per API call (100 is the maximum per call)
    max_total=200,   # CHANGE TO ANY NUMBER OF VIDEOS YOU WANT TO COLLECT IN TOTAL
)

In [ ]:
# Run the query — fetch_all_pages=True handles pagination automatically
videos, search_id, cursor, has_more, start_used, end_used = api.query_videos(
    request, 
    fetch_all_pages=True
)
print(f"Collected {len(videos)} videos from @{username}")

In [ ]:
# Save to CSV
df = pd.DataFrame([{f: v.get(f) for f in VIDEO_FIELDS} for v in videos])
df["scraped_at"] = datetime.now(timezone.utc)

# Naming convention: tool_endpoint_target_timestamp.csv
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
outfile = f"api_user_{username}_{timestamp}.csv"

df.to_csv(outfile, index=False)
print(f"Saved {len(df)} videos → {outfile}")

### Querying User Info (Separate Endpoint)

The video endpoint **does not** return user-level details like display name, bio, follower count, or verification status. The fields you got back in 2-1a are limited to what's documented under [Query Videos](https://developers.tiktok.com/doc/research-api-specs-query-videos). For richer user metadata you have to call the **user info** endpoint separately:

In [ ]:
# Query user info — separate endpoint, separate API call.
user_info_request = QueryUserInfoRequest(
    username=username,   # reusing the same username from 2-1a
)
user_info = api.query_user_info(user_info_request)
print(user_info)

**This is a real limitation of the Research API.** Getting user metadata alongside video metadata requires **two** endpoints, and each one counts against your daily quota. If you want enriched data for every user in a 1,000-video collection, that's potentially 1,000 extra calls just for the user details.

Pyktok (Section 2-2) and Apify (Section 2-3) don't have this issue — both return user metadata **embedded** in the video objects they scrape, so a single call gives you both. That's one of the trade-offs in the API-vs-scraping comparison.

### 2-1b. Keyword Search

The keyword endpoint searches across TikTok for videos containing a term. We use the same `api` client and `VIDEO_FIELDS` from above — just change the query field from `username` to `keyword`.

Start with a single straightforward query, then we'll look at how to keep going if it returns `has_more=True`.

In [ ]:
# CHANGE KEYWORD TO YOUR OWN SEARCH TERM
keyword = "climate change"

# CHANGE START_DATE TO YOUR EARLIEST DATE (YYYYMMDD) — both start_date and end_date are REQUIRED by the API
start_date = "20260401"
# CHANGE END_DATE TO YOUR LATEST DATE (YYYYMMDD)
end_date   = "20260430"

query = Query(and_criteria=[Criteria("EQ", "keyword", [keyword])])

In [ ]:
# A simple single-window query.
request = QueryVideoRequest(
    fields=",".join(VIDEO_FIELDS),
    query=query,
    start_date=start_date,
    end_date=end_date,
    max_count=100,   # videos per API call (100 is the maximum)
    max_total=100,  # CHANGE MAX_TOTAL TO YOUR DESIRED COUNT (TikTok caps each query)
)

videos, search_id, cursor, has_more, start_date, end_date = api.query_videos(
    request,
    fetch_all_pages=True,
)
print(f"Collected {len(videos)} videos for '{keyword}'")
print(f"has_more = {has_more}") # True if there are more results within this date range

In [ ]:
# Save to CSV
df = pd.DataFrame([{f: v.get(f) for f in VIDEO_FIELDS} for v in videos])
df["scraped_at"]    = datetime.now(timezone.utc)
df["query_keyword"] = keyword

timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
outfile   = f"api_keyword_{keyword.replace(' ', '_')}_{timestamp}.csv"
df.to_csv(outfile, index=False)
print(f"Saved {len(df)} videos → {outfile}")

**Watch out for high-volume terms.** For viral keywords like `viral`, `fyp`, or trending topics, the API often returns only the most recent **~2 minutes** of activity even when you ask for a full month — TikTok caps the response to the freshest activity (Pearson et al. 2025; Jeon et al. 2026).

You can't predict in advance how dense a window is, so setting `start_date` / `end_date` is a guess. A bad guess means missed data and wasted quota.

If your previous call returned `has_more = True`, you can resume **within the same date range** by passing the `search_id` and `cursor` that came back in the previous response:

In [ ]:
# Continue from where we stopped — same date range, just resume.
request = QueryVideoRequest(
    fields=",".join(VIDEO_FIELDS),
    query=query,
    start_date=start_date,    # value returned by the previous call
    end_date=end_date,         # value returned by the previous call
    max_count=100,
    max_total=100,
    search_id=search_id,       # from previous response (resume token)
    cursor=cursor,             # from previous response (where we stopped)
)

videos, search_id, cursor, has_more, start_date, end_date = api.query_videos(
    request,
    fetch_all_pages=True,
)
print(f"Collected {len(videos)} more videos (continuing the same session)")

### 2-1c. Hashtag Search

Hashtag queries are identical to keyword queries except for one field name — `hashtag_name` instead of `keyword`:

```python
query = Query(and_criteria=[Criteria("EQ", "hashtag_name", [hashtag])])
```

The `search_id` / `cursor` continuation from 2-1b lets you resume **within a date range**, but it can't change the date window itself. If you want to **extend the time period** — to manage quota better or to keep going until you hit a target count — you need to **slide the date window** itself.

The trick: keep each window **tight** (e.g. 2 days) so a single window doesn't saturate the API response, then slide it backward in time until you've collected enough. This is especially useful for high-volume hashtags where a wide window only returns the most recent minutes of activity.

In [ ]:
# CHANGE HASHTAG TO YOUR OWN TARGET (no '#')
hashtag = "booktok"

# CHANGE MAX_TOTAL TO YOUR DESIRED COUNT
max_total = 100

# CHANGE WINDOW_DAYS TO A SHORTER WINDOW FOR HIGH-VOLUME TAGS (1-2 for viral; 7-30 for niche)
window_days = 2

query = Query(and_criteria=[Criteria("EQ", "hashtag_name", [hashtag])])

In [ ]:
# Sliding window: start today, collect a window, slide back, repeat until we hit max_total.
all_videos = []
end_date   = datetime.now(timezone.utc).strftime("%Y%m%d")
start_date = (datetime.now(timezone.utc) - timedelta(days=window_days)).strftime("%Y%m%d")

while len(all_videos) < max_total:
    request = QueryVideoRequest(
        fields=",".join(VIDEO_FIELDS),
        query=query,
        start_date=start_date,
        end_date=end_date,
        max_count=100,
        max_total=max_total - len(all_videos),   # only ask for what's still missing
    )
    videos, _, _, _, _, _ = api.query_videos(request, fetch_all_pages=True)
    all_videos.extend(videos)
    print(f"  {start_date}–{end_date}: +{len(videos)} → {len(all_videos)} total")

    # Slide the window back by window_days
    end_date   = start_date
    start_date = (datetime.strptime(start_date, "%Y%m%d") - timedelta(days=window_days)).strftime("%Y%m%d")
    if int(start_date[:4]) < 2018:
        print("Reached 2018 floor. Stopping.")
        break

print(f"\nTotal: {len(all_videos)} videos for #{hashtag}")

In [ ]:
# Save to CSV
df = pd.DataFrame([{f: v.get(f) for f in VIDEO_FIELDS} for v in all_videos])
df["scraped_at"]    = datetime.now(timezone.utc)
df["query_hashtag"] = hashtag

timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
outfile   = f"api_hashtag_{hashtag}_{timestamp}.csv"
df.to_csv(outfile, index=False)
print(f"Saved {len(df)} videos → {outfile}")

### 2-1d. Comments

The comment endpoint takes a **video ID** (not a URL) and returns up to 100 comments per call. You'll usually grab the video IDs from a user CSV you already collected in 2-1a.

In [ ]:
# CHANGE VIDEO_ID TO ANY PUBLIC TIKTOK VIDEO ID — you can re-use one from 2-1a:
#   video_id = videos[0]['id']
video_id = videos[0]['id']

# Comment fields available from the Research API.
# Reference: https://developers.tiktok.com/doc/research-api-specs-query-video-comments
COMMENT_FIELDS = ["id", "text", "like_count", "reply_count", "parent_comment_id", "create_time"]

In [ ]:
video_id

In [ ]:
# A simple single-page comment query.
request = QueryVideoCommentsRequest(
    video_id=video_id,
    max_count=100,   # comments per API call (100 is the maximum)
)
comments, cursor, has_more = api.query_video_comments(request, fetch_all_pages=False)

print(f"Collected {len(comments)} comments for video {video_id}")
print(f"has_more = {has_more}   # True if there are more comment pages to fetch")

In [ ]:
# Save to CSV
df = pd.DataFrame([{f: c.get(f) for f in COMMENT_FIELDS} for c in comments])
df["video_id"]   = video_id
df["scraped_at"] = datetime.now(timezone.utc)

timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
outfile   = f"api_comment_{video_id}_{timestamp}.csv"
df.to_csv(outfile, index=False)
print(f"Saved {len(df)} comments → {outfile}")

If `has_more` was `True`, fetch the next page by passing back the `cursor` from the previous response — same pattern as the keyword continuation in 2-1b:

In [ ]:
# Continue from where we stopped — same video_id, just resume.
request = QueryVideoCommentsRequest(
    video_id=video_id,
    max_count=100,  
    cursor=cursor,    # from previous response (where we stopped)
)
comments, cursor, has_more = api.query_video_comments(request, fetch_all_pages=False)
print(f"Collected {len(comments)} more comments (continuing the same session)")

### 2-1e. Token Auto-Renewal (For Long-Running Collection)

If you plan to run collection that exceeds two hours in a single session, the API client above will stop working once the token expires. You won't hit this during the tutorial, but it matters for any real data collection project.

The `TokenManager` class below handles renewal automatically: before each request it checks whether the token is still valid (with a 10-minute buffer), and silently fetches a new one if not. Swap it in by replacing the `api = TikTokResearchAPI(...)` line with the two lines at the bottom of this cell.

In [ ]:
import requests
from datetime import datetime, timezone, timedelta

class TokenManager:
    """Keeps your TikTok API token alive across sessions longer than 2 hours."""

    def __init__(self, credential_file="credential.env"):
        load_dotenv(credential_file, override=True)
        self.client_key    = os.getenv("CLIENT_KEY")
        self.client_secret = os.getenv("CLIENT_SECRET")
        self.token         = None
        self.expires_at    = None

    def get_api_client(self, qps=2):
        # Refresh the token if it's expired or missing
        if not self.token or datetime.now(timezone.utc) >= self.expires_at:
            self._refresh()
        return TikTokResearchAPI(
            client_key=self.client_key,
            client_secret=self.client_secret,
            qps=qps,
        )

    def _refresh(self):
        resp = requests.post(
            "https://open.tiktokapis.com/v2/oauth/token/",
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            data={
                "client_key":    self.client_key,
                "client_secret": self.client_secret,
                "grant_type":    "client_credentials",
            },
            timeout=30,
        )
        resp.raise_for_status()
        data        = resp.json()
        self.token  = data["access_token"]
        # Subtract 10 minutes so we refresh before it actually expires
        self.expires_at = datetime.now(timezone.utc) + timedelta(seconds=data["expires_in"] - 600)
        print(f"  Token refreshed. Valid until {self.expires_at.strftime('%H:%M UTC')}")

# To use it: replace the simple setup above with these two lines
# token_mgr = TokenManager()
# api = token_mgr.get_api_client()

---
## 2-2. Apify

Apify is a commercial cloud scraping platform. For TikTok, we use two pre-built actors (Apify's term for scraping bots): `clockworks/tiktok-scraper` for videos and profiles, and `clockworks/tiktok-comments-scraper` for comments.

Like Pyktok, Apify scrapes the front-end — it captures the algorithmically curated view of TikTok, not a back-end database. The main practical difference: Apify runs in the cloud, so you don't need to keep your laptop open. You kick off a job, it runs remotely, and you pull the results when it's done.

**Cost:** The free tier gives you about $5 in credits. A typical small collection run (a few hundred results) costs \$0.10–\$0.50. Large runs (50K videos) can cost a few dollars. Keep an eye on your usage in the Apify console.

Apify uses async Python, so you'll see `await` in the code below. In Jupyter, you can call `await` directly in cells.

In [ ]:
import asyncio
import os
import glob
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from pandas import json_normalize
from dotenv import load_dotenv
from apify_client import ApifyClientAsync

load_dotenv("credential.env", override=True)

def get_apify_client():
    """Create an Apify async client from your stored token."""
    token = os.getenv("APIFY_API_TOKEN")
    if not token:
        raise ValueError("APIFY_API_TOKEN not found in credential.env")
    return ApifyClientAsync(token=token)

In [ ]:
async def stream_to_csv(dataset, outfile, extra_fields=None):
    """
    Stream results from an Apify dataset to a CSV file, row by row.
    json_normalize flattens nested fields (e.g., 'author.name' becomes 'author_name').
    """
    count          = 0
    header_written = Path(outfile).exists() and Path(outfile).stat().st_size > 0

    async for item in dataset.iterate_items():
        if extra_fields:
            item.update(extra_fields)
        df = json_normalize(item, sep="_", max_level=2)
        df["scraped_at"] = datetime.now(timezone.utc)
        df.to_csv(outfile, mode="a", header=not header_written, index=False)
        header_written = True
        count += 1

    return count

### 2-2a. User Videos

The Apify user scraper takes a username and collects their recent videos. `resultsPerPage` caps the total returned. Results stream in as the actor runs — we save them to CSV as they arrive.

In [ ]:
# CHANGE USERNAME AND MAX_RESULTS TO YOUR OWN TARGET ACCOUNT AND DESIRED COUNT.
username    = "apnews"
max_results = 5

In [ ]:
async def collect_user_apify(username, max_results=50):
    client    = get_apify_client()
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    outfile   = f"apify_user_{username}_{timestamp}.csv"

    run_input = {
        "profiles":              [username],
        "resultsPerPage":        max_results,
        "profileSorting":        "latest",        # newest videos first
        "excludePinnedPosts":    True,            # pinned posts can skew recency
        "shouldDownloadVideos":  False,           # metadata only
        "shouldDownloadCovers":  False,
        "profileScrapeSections": ["videos"],
    }

    print(f"Starting Apify run for @{username}...")
    run     = await client.actor("clockworks/tiktok-scraper").call(run_input=run_input)
    dataset = client.dataset(run.get("defaultDatasetId"))
    count   = await stream_to_csv(dataset, outfile)

    print(f"Saved {count} videos → {outfile}")
    return outfile

# Run it
apify_user_file = await collect_user_apify(username, max_results=max_results)

In [ ]:
# Preview the output
# Apify returns a lot of fields — let's see what we got
df = pd.read_csv(apify_user_file)
print(f"Collected {len(df)} videos  ({len(df.columns)} fields per video)")
print("\nFirst few column names:")
print(df.columns.tolist()[:15])

### 2-2b. Keyword Search

Same actor, different input: swap `profiles` for `searchQueries`. Apify manages pagination internally, so you just set `resultsPerPage` and let it run.

In [ ]:
# CHANGE KEYWORD AND MAX_RESULTS TO YOUR OWN SEARCH TERM AND DESIRED COUNT.
keyword     = "grwm"
max_results = 5

In [ ]:
async def collect_keyword_apify(keyword, max_results=500):
    client    = get_apify_client()
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    outfile   = f"apify_keyword_{keyword.replace(' ', '_')}_{timestamp}.csv"

    run_input = {
        "searchQueries":         [keyword],
        "resultsPerPage":        max_results,
        "shouldDownloadVideos":  False,
        "shouldDownloadCovers":  False,
        "newestPostDate":        datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    }

    print(f"Starting Apify run for keyword '{keyword}'...")
    run     = await client.actor("clockworks/tiktok-scraper").call(run_input=run_input)
    dataset = client.dataset(run.get("defaultDatasetId"))
    count   = await stream_to_csv(dataset, outfile, extra_fields={"query_keyword": keyword})

    print(f"Saved {count} videos → {outfile}")
    return outfile

outfile = await collect_keyword_apify(keyword, max_results=max_results)

### 2-2c. Hashtag Search

Same pattern again — just swap `searchQueries` for `hashtags`. No `#` in the string.

In [ ]:
# CHANGE HASHTAG AND MAX_RESULTS TO YOUR OWN TARGET (no '#') AND DESIRED COUNT.
hashtag     = "foodtok"
max_results = 5

In [ ]:
async def collect_hashtag_apify(hashtag, max_results=500):
    client    = get_apify_client()
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    outfile   = f"apify_hashtag_{hashtag}_{timestamp}.csv"

    run_input = {
        "hashtags":              [hashtag],   # no '#' needed here
        "resultsPerPage":        max_results,
        "shouldDownloadVideos":  False,
        "shouldDownloadCovers":  False,
        "newestPostDate":        datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    }

    print(f"Starting Apify run for #{hashtag}...")
    run     = await client.actor("clockworks/tiktok-scraper").call(run_input=run_input)
    dataset = client.dataset(run.get("defaultDatasetId"))
    count   = await stream_to_csv(dataset, outfile, extra_fields={"query_hashtag": hashtag})

    print(f"Saved {count} videos → {outfile}")
    return outfile

outfile = await collect_hashtag_apify(hashtag, max_results=max_results)

### 2-2d. Comments

Apify uses a separate actor for comments: `clockworks/tiktok-comments-scraper`. It takes video URLs (not IDs), and returns up to `commentsPerPost` top-level comments. We'll pull the URLs from the user collection we ran in 2-2a.

In [ ]:
def extract_video_urls_apify(user_csv, limit=5):
    """Pull video URLs from an Apify user collection CSV."""
    df   = pd.read_csv(user_csv)
    urls = []

    for _, row in df.iterrows():
        # Apify stores the full video URL in 'webVideoUrl'
        if pd.notna(row.get("webVideoUrl")):
            video_id = str(row.get("id", "unknown"))
            urls.append((video_id, row["webVideoUrl"]))

    return urls[:limit]

In [ ]:
async def collect_comments_apify(username, video_urls, comments_per_video=1):
    """Collect comments for each video URL via Apify's comment scraper."""
    client = get_apify_client()

    for video_id, video_url in video_urls:
        timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
        outfile   = f"apify_comment_{username}_{video_id}_{timestamp}.csv"

        print(f"Fetching comments for {video_id}...")
        run_input = {
            "postURLs":           [video_url],
            "commentsPerPost":    comments_per_video,
            "maxRepliesPerComment": 0,  # top-level comments only
        }

        run     = await client.actor("clockworks/tiktok-comments-scraper").call(run_input=run_input)
        dataset = client.dataset(run.get("defaultDatasetId"))
        count   = await stream_to_csv(dataset, outfile, extra_fields={"video_id": video_id})

        print(f"  Saved {count} comments → {outfile}")

        # Brief pause between videos
        await asyncio.sleep(10)

In [ ]:
# Load video URLs from the Apify user collection
user_files = glob.glob(f"apify_user_{username}_*.csv")

if user_files:
    video_urls = extract_video_urls_apify(max(user_files), limit=5)
    print(f"Found {len(video_urls)} video URLs")
    await collect_comments_apify(username, video_urls, comments_per_video=500)
else:
    print("Run the user collection first (Section 2-2a)")

### 2-2e. Related Videos

Apify's `clockworks/tiktok-scraper` actor supports related video collection by enabling the `scrapeRelatedVideos` flag and passing video URLs. This gives you the same "You may like" stream as Pyktok, but running in the cloud. We tag each result with the parent video URL so you can trace the recommendation chain.

In [ ]:
async def collect_related_apify(username, video_urls, count_per_video=10):
    """Collect TikTok's 'You may like' stream for each seed video."""
    client = get_apify_client()

    for video_id, video_url in video_urls:
        timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
        outfile   = f"apify_related_{username}_{video_id}_{timestamp}.csv"

        print(f"Fetching related videos for {video_id}...")
        run_input = {
            "postURLs":               [video_url],
            "scrapeRelatedVideos":    True,         # this is the key flag
            "resultsPerPage":         count_per_video,
            "shouldDownloadVideos":   False,
            "shouldDownloadCovers":   False,
        }

        run     = await client.actor("clockworks/tiktok-scraper").call(run_input=run_input)
        dataset = client.dataset(run.get("defaultDatasetId"))
        count   = await stream_to_csv(
            dataset, outfile,
            extra_fields={"parent_video_url": video_url, "parent_video_id": video_id},
        )

        print(f"  Saved {count} related videos → {outfile}")
        await asyncio.sleep(10)

In [ ]:
# Use the same video URLs from Section 2-2d
if user_files:
    video_urls = extract_video_urls_apify(max(user_files), limit=1)
    await collect_related_apify(username, video_urls, count_per_video=5)
else:
    print("Run the user collection first (Section 2-2a)")

---
## 2-3. Pyktok

Pyktok is an open-source Python library that uses browser automation (Playwright) to scrape TikTok's public endpoints. Unlike the Research API which queries TikTok's backend directly, Pyktok captures the front-end experience — the algorithmically curated view that users see.

**Key features:**
- No API credentials needed for public endpoints
- Returns rich user metadata embedded in video objects (no separate calls)
- Can run with or without login (login enables more endpoints)
- Free and open source

**Available endpoints:**

| Endpoint | Function | Login Required |
|---|---|---|
| User info | `pyk.get_user_info(username)` | Optional |
| User videos | `pyk.get_user_videos(username, count)` | Recommended |
| Hashtag videos | `pyk.get_hashtag_videos(hashtag, count)` | No |
| Keyword search | `pyk.search_videos(keyword, count)` | No |
| Sound videos | `pyk.get_sound_videos(sound_id, count)` | No |
| Related videos | `pyk.get_related_videos(url, count)` | Yes |
| Comments | `pyk.get_video_comments(url, count)` | Yes |
| Trending | `pyk.get_trending_videos(count, region)` | Yes |

### Setup: Start a Browser Session

Import pyktok and initialize a browser. You can run in headless mode (no visible window) or with the browser visible for debugging.

In [ ]:
import pyktok_2026 as pyk
import pandas as pd
from IPython.display import display

# Initialize browser - set headless=False to see the browser window
pyk.specify_browser('chrome', headless=True)

print('pyktok ready')

### (Optional) Login with Session ID

For endpoints that require or benefit from login (user videos, comments, related videos), you can authenticate using a sessionid cookie:

1. Open TikTok in your browser while logged in
2. Open DevTools → Application/Storage → Cookies → www.tiktok.com
3. Copy the value of the `sessionid` cookie
4. Paste it below

**Note:** The no-login endpoints (hashtag, keyword, sound) work without this step.

In [ ]:
# pyk.login_with_cookies(sessionid='paste your sessionid here')
# pyk.login_status(verify_signing=True)
# pyk.healthcheck()

In [ ]:
# Load sessionid from credential.env
import os
from dotenv import load_dotenv

load_dotenv("credential.env", override=True)
sessionid = os.getenv("SESSIONID")

if sessionid:
    pyk.login_with_cookies(sessionid=sessionid)
    pyk.login_status(verify_signing=True)
    pyk.healthcheck()
    print(f"Logged in with sessionid from credential.env")
else:
    print("No SESSIONID found in credential.env - running without login")

### Define Your Targets

Set the username, hashtag, keyword, sound_id, and URL you want to collect data from.

In [ ]:
username = 'badbunny'
    # username = 'alexismarievans'
    # username = 'ariellelorre'
    # username = 'tiktok'
    # username = 'barrettplasticsurgery'
    # username = 'bobbyparrish'
    # username = 'charlieputh'
    # username = 'dietitianwithtwins'
    # username = 'dixiedamelio'
    # username = 'dzaslavsky'
    # username = 'fifaworldcup'
    # username = 'katteyes'
    # username = 'knick_nat'
    # username = 'kristincavallari'
    # username = 'PracticeByPalmer'
    # username = 'realdonaldtrump'
    # username = 'realmadrid'
    # username = 'robertfkennedyjrofficial'
    # username = 'sam.shan.shops'
    # username = 'wired'
    # username = 'zohran_k_mamdani'

hashtag  = 'fyp'
    # hashtag  = 'aivideos'
    # hashtag  = 'blackowned'
    # hashtag  = 'buffalo'
    # hashtag  = 'coke'
    # hashtag  = 'elnino'
    # hashtag  = 'grwm'
    # hashtag  = 'ice'
    # hashtag  = 'momlife'
    # hashtag  = 'relatable'
    # hashtag  = 'switzerland'
    # hashtag  = 'wegovy'

keyword  = 'news'
    # keyword  = 'botox'
    # keyword  = 'fifaworldcup'
    # keyword  = 'heavenfirsthealing'
    # keyword  = 'put sunscreen on'
    # keyword  = 'spain'
    # keyword  = 'waymo'

sound_id = '6705081440117196802'
    # sound_id = '6602666381370460933'
    # sound_id = '6704852053862123521'
    # sound_id = '6731131741387360258'
    # sound_id = '6917178230261614593'
    # sound_id = '6917632754804852737'
    # sound_id = '6964415885571033090'
    # sound_id = '7005888397580208130'
    # sound_id = '7099827699635505963'
    # sound_id = '7211414788142794754'
    # sound_id = '7265149627312229163'
    # sound_id = '7423028069247601451'
    # sound_id = '7500208195948792618'
    # sound_id = '7528431704882154271'
    # sound_id = '7637147165290924831'

url  = 'https://www.tiktok.com/@fifaworldcup/video/7664597780987481366'
    # url  = 'https://www.tiktok.com/@adidas/video/7664397211828030722'
    # url  = 'https://www.tiktok.com/@alexismarievans/video/7168920554664135979'
    # url  = 'https://www.tiktok.com/@ariellelorre/video/7389440975392869675'
    # url  = 'https://www.tiktok.com/@badbunny/video/7463871059796806954'
    # url  = 'https://www.tiktok.com/@badbunny/video/7660593958518639886'
    # url  = 'https://www.tiktok.com/@barrettplasticsurgery/video/7420089405794225439'
    # url  = 'https://www.tiktok.com/@bo.bap.kids/video/7663316863039966484'
    # url  = 'https://www.tiktok.com/@bootsandbales67/video/7632564058453019911'
    # url  = 'https://www.tiktok.com/@brainbodybyjuless/video/7657640105531936014'
    # url  = 'https://www.tiktok.com/@charlieputh/video/7008700077309054213'
    # url  = 'https://www.tiktok.com/@deziah4u/video/7662825760159288598'
    # url  = 'https://www.tiktok.com/@dzaslavsky/video/7656085552986426655'
    # url  = 'https://www.tiktok.com/@fersitamuak/video/7662524840498285831'
    # url  = 'https://www.tiktok.com/@fifaworldcup/photo/7664365127990299926'
    # url  = 'https://www.tiktok.com/@fifaworldcup/video/7656298830769474838'
    # url  = 'https://www.tiktok.com/@fifaworldcup/video/7656298991348469014'
    # url  = 'https://www.tiktok.com/@fifaworldcup/video/7656312849656040726'
    # url  = 'https://www.tiktok.com/@fifaworldcup/video/7664363237894376726'
    # url  = 'https://www.tiktok.com/@jayden26681/video/7664585480599653646'
    # url  = 'https://www.tiktok.com/@kalabeautypalace/video/7665844369882172685'
    # url  = 'https://www.tiktok.com/@knick_nat/video/7622359981735677214'
    # url  = 'https://www.tiktok.com/@koujj08/video/7640800405563247885'
    # url  = 'https://www.tiktok.com/@lapheaduh/video/7640924310353333534'
    # url  = 'https://www.tiktok.com/@luckybaby_58/video/7644520746148056351'
    # url  = 'https://www.tiktok.com/@nicolas_williams9/video/7664406384649276694'
    # url  = 'https://www.tiktok.com/@officalpicklepesh/video/7647128625657023774'
    # url  = 'https://www.tiktok.com/@sam.shan.shops/video/7618363708313537813'
    # url  = 'https://www.tiktok.com/@tiktok/video/7106594312292453675'
    # url  = 'https://www.tiktok.com/@usman.plyaz/video/7663878975936089365'
    # url = 'https://www.tiktok.com/@badbunny/video/7660593958518639886'
    # url = 'https://www.tiktok.com/@adidas/video/7664397211828030722'
    # url = 'https://www.tiktok.com/@alexismarievans/video/7168920554664135979'
    # url = 'https://www.tiktok.com/@ariellelorre/video/7389440975392869675'
    # url = 'https://www.tiktok.com/@badbunny/video/7463871059796806954'
    # url = 'https://www.tiktok.com/@barrettplasticsurgery/video/7420089405794225439'
    # url = 'https://www.tiktok.com/@bo.bap.kids/video/7663316863039966484'
    # url = 'https://www.tiktok.com/@bootsandbales67/video/7632564058453019911'
    # url = 'https://www.tiktok.com/@brainbodybyjuless/video/7657640105531936014'
    # url = 'https://www.tiktok.com/@charlieputh/video/7008700077309054213'
    # url = 'https://www.tiktok.com/@deziah4u/video/7662825760159288598'
    # url = 'https://www.tiktok.com/@dzaslavsky/video/7656085552986426655'
    # url = 'https://www.tiktok.com/@fersitamuak/video/7662524840498285831'
    # url = 'https://www.tiktok.com/@fifaworldcup/photo/7664365127990299926'
    # url = 'https://www.tiktok.com/@fifaworldcup/video/7656298830769474838'
    # url = 'https://www.tiktok.com/@fifaworldcup/video/7656298991348469014'
    # url = 'https://www.tiktok.com/@fifaworldcup/video/7656312849656040726'
    # url = 'https://www.tiktok.com/@fifaworldcup/video/7664363237894376726'
    # url = 'https://www.tiktok.com/@fifaworldcup/video/7664597780987481366'
    # url = 'https://www.tiktok.com/@jayden26681/video/7664585480599653646'
    # url = 'https://www.tiktok.com/@kalabeautypalace/video/7665844369882172685'
    # url = 'https://www.tiktok.com/@knick_nat/video/7622359981735677214'
    # url = 'https://www.tiktok.com/@koujj08/video/7640800405563247885'
    # url = 'https://www.tiktok.com/@lapheaduh/video/7640924310353333534'
    # url = 'https://www.tiktok.com/@luckybaby_58/video/7644520746148056351'
    # url = 'https://www.tiktok.com/@nicolas_williams9/video/7664406384649276694'
    # url = 'https://www.tiktok.com/@officalpicklepesh/video/7647128625657023774'
    # url = 'https://www.tiktok.com/@sam.shan.shops/video/7618363708313537813'
    # url = 'https://www.tiktok.com/@tiktok/video/7106594312292453675'
    # url = 'https://www.tiktok.com/@usman.plyaz/video/7663878975936089365'

### 2-3a. User Info

In [ ]:
print(pyk.get_user_info(username))

### 2-3b. User Videos

In [ ]:
from datetime import datetime, timezone

df = pd.DataFrame(pyk.get_user_videos(username, count=20, timeout=90))
display(df)

# Save to CSV
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
outfile = f"pyktok_user_{username}_{timestamp}.csv"
df.to_csv(outfile, index=False)
print(f"Saved {len(df)} videos → {outfile}")

### 2-3d. Hashtag Info

In [ ]:
print(pyk.get_hashtag_info(hashtag))

### 2-3e. Hashtag Videos

In [ ]:
from datetime import datetime, timezone

df = pd.DataFrame(pyk.get_hashtag_videos(hashtag, count=20, timeout=90))
display(df)

# Save to CSV
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
outfile = f"pyktok_hashtag_{hashtag}_{timestamp}.csv"
df.to_csv(outfile, index=False)
print(f"Saved {len(df)} videos → {outfile}")

### 2-3f. Sound Info

In [ ]:
print(pyk.get_sound_info(sound_id))

In [ ]:
pyk.close()
print('Pyktok session closed.')

### Close Browser Session

When you're done collecting data, close the browser to free up resources.

In [ ]:
from datetime import datetime, timezone

df = pd.DataFrame(pyk.search_videos(keyword, count=20, timeout=90))
display(df)

# Save to CSV
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
outfile = f"pyktok_keyword_{keyword.replace(' ', '_')}_{timestamp}.csv"
df.to_csv(outfile, index=False)
print(f"Saved {len(df)} videos → {outfile}")

### 2-3z. Troubleshooting

In [ ]:
# playlist_id = 'your_playlist_id_here'
# display(pd.DataFrame(pyk.get_playlist_videos(playlist_id, count=20)))

### 2-3y. Playlist Videos (if applicable)

- If a cell hangs, run `pyk.reset()` then `pyk.specify_browser('chrome')` again
- `pyk.close()` shuts the browser and cleans up resources

### 2-3x. Multi-Page with MP4 Download (Heavy)

In [ ]:
pyk.save_tiktok_multi_page(sound_id, ent_type='sound', video_ct=20)

### 2-3w. Multi-Page Sound Feed

In [ ]:
pyk.save_tiktok_multi_page(url, ent_type='video_related', video_ct=20)

### 2-3v. Multi-Page Related Feed

In [ ]:
pyk.save_tiktok_multi_page(hashtag, ent_type='hashtag', video_ct=20)

### 2-3u. Multi-Page Hashtag Feed

In [ ]:
pyk.save_tiktok_multi_page(username, ent_type='user', video_ct=20)

### 2-3t. Multi-Page User Feed

### Multi-Page Collection Functions

These functions collect larger datasets by paginating through feeds.

In [ ]:
pyk.save_user(username, count=20)

### 2-3c. Save User Videos (Alternative using yt-dlp)

In [ ]:
pyk.download_video(url, dir_path='clips')

### 2-3s. Download MP4 Only

In [ ]:
url2 = 'https://www.tiktok.com/@tiktok/video/7106594312292453675'
pyk.save_tiktok_multi_urls([url, url2])

### 2-3r. Save Multiple URLs

In [ ]:
pyk.save_tiktok(url, save_video=True, dir_path='clips')

### 2-3q. Save Video Metadata + MP4

In [ ]:
from datetime import datetime, timezone

# Note: Run the "Video Comments" cell first to populate cmts
df = pd.DataFrame(pyk.get_comment_replies(cmts[0]['cid'], url, count=50))
display(df)

# Save to CSV
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
video_id = url.split('/')[-1]
outfile = f"pyktok_comment_replies_{video_id}_{cmts[0]['cid']}_{timestamp}.csv"
df.to_csv(outfile, index=False)
print(f"Saved {len(df)} comment replies → {outfile}")

### 2-3p. Comment Replies (Login Required)

In [ ]:
pyk.save_tiktok_comments(url, comment_count=100, timeout=180)

### 2-3n. Get Comments as DataFrame (Login Required)

In [ ]:
display(pyk.get_tiktok_comments(url, comment_count=30))

### 2-3m. Video Comments (Login Required)

In [ ]:
from datetime import datetime, timezone

cmts = pyk.get_video_comments(url, count=50)
df = pd.DataFrame(cmts)
display(df)

# Save to CSV
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
video_id = url.split('/')[-1]
outfile = f"pyktok_comments_{video_id}_{timestamp}.csv"
df.to_csv(outfile, index=False)
print(f"Saved {len(df)} comments → {outfile}")

### 2-3l. Related Videos (Login Required)

In [ ]:
from datetime import datetime, timezone

df = pd.DataFrame(pyk.get_related_videos(url, count=16))
display(df)

# Save to CSV
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
video_id = url.split('/')[-1]
outfile = f"pyktok_related_{video_id}_{timestamp}.csv"
df.to_csv(outfile, index=False)
print(f"Saved {len(df)} related videos → {outfile}")

### 2-3k. Trending Videos (Login Required)

In [ ]:
from datetime import datetime, timezone

df = pd.DataFrame(pyk.get_trending_videos(count=20, region='US'))
display(df)

# Save to CSV
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
outfile = f"pyktok_trending_US_{timestamp}.csv"
df.to_csv(outfile, index=False)
print(f"Saved {len(df)} trending videos → {outfile}")

### 2-3j. Search Hashtags

In [ ]:
from datetime import datetime, timezone

df = pd.DataFrame(pyk.search_hashtags(keyword, count=10))
display(df)

# Save to CSV
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
outfile = f"pyktok_search_hashtags_{keyword.replace(' ', '_')}_{timestamp}.csv"
df.to_csv(outfile, index=False)
print(f"Saved {len(df)} hashtags → {outfile}")

### 2-3i. Search Users

In [ ]:
from datetime import datetime, timezone

df = pd.DataFrame(pyk.search_users(keyword, count=10))
display(df)

# Save to CSV
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
outfile = f"pyktok_search_users_{keyword.replace(' ', '_')}_{timestamp}.csv"
df.to_csv(outfile, index=False)
print(f"Saved {len(df)} users → {outfile}")

### 2-3h. Keyword Search

### 2-3g. Sound Videos

In [ ]:
from datetime import datetime, timezone

df = pd.DataFrame(pyk.get_sound_videos(sound_id, count=20, timeout=90))
display(df)

# Save to CSV
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
outfile = f"pyktok_sound_{sound_id}_{timestamp}.csv"
df.to_csv(outfile, index=False)
print(f"Saved {len(df)} videos → {outfile}")

---
## 3. Exploring Your Data

The cells below walk through a small descriptive analysis of the CSVs
shipped in [`sample_data/`](sample_data) — four pyktok user-video
collections (`@aaronparnas1`, `@apnews`, `@netflix`, `@realdonaldtrump`),
eight comment files spanning a handful of videos by those users, plus
an aggregated hashtag file. The same code runs on the CSVs you generate
in sections 2-1 to 2-3 — just point the file globs at your `output/`
directory instead.

What we look at:

- **3.1 Load + inspect** — read every user CSV in `sample_data/`,
  combine them, check shape and dtypes.
- **3.2 Video-level distributions** — duration, upload dates, engagement.
- **3.3 Per-user aggregates** — videos per user, total duration per user.
- **3.4 Comments analysis** — comment volume, length, language, top
  commenters, reply patterns.


### 3.1 Load + inspect

In [ ]:
import glob
import ast
from pathlib import Path
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 30)

# CHANGE SAMPLE_DIR TO POINT AT YOUR OWN COLLECTION FOLDER
# (the default reads from the sample CSVs that ship with this repo).
SAMPLE_DIR = Path("sample_data")

# CHANGE USER_GLOB / COMMENT_GLOB TO MATCH YOUR FILENAMES.
# Default conventions: pyktok_user_<handle>_<timestamp>.csv
#                     pyktok_comment_<handle>_<videoid>_<timestamp>.csv
USER_GLOB    = "pyktok_user_*.csv"
COMMENT_GLOB = "pyktok_comment_*.csv"

user_files    = sorted(SAMPLE_DIR.glob(USER_GLOB))
comment_files = sorted(SAMPLE_DIR.glob(COMMENT_GLOB))

print(f"User CSVs found    : {len(user_files)}")
for p in user_files:    print(f"   {p.name}")
print(f"\nComment CSVs found : {len(comment_files)}")
for p in comment_files: print(f"   {p.name}")


In [ ]:
# Load + concatenate every user CSV into one DataFrame.
videos = pd.concat([pd.read_csv(p) for p in user_files], ignore_index=True)

# Parse the ISO timestamp once so the rest of the section can use it.
videos["video_timestamp"] = pd.to_datetime(videos["video_timestamp"], errors="coerce")

print(f"Combined: {len(videos):,} videos x {videos.shape[1]} columns")
print(f"Users   : {videos['author_username'].nunique()}  "
      f"({sorted(videos['author_username'].dropna().unique())})")
print(f"Date range : {videos['video_timestamp'].min()} -> {videos['video_timestamp'].max()}")

videos.head(3)


In [ ]:
# Column names + dtypes — useful when you swap in your own files.
videos.dtypes

In [ ]:
# Numeric summary for the engagement + duration columns.
ENGAGEMENT_COLS = [
    "video_duration",
    "video_playcount", "video_diggcount",
    "video_commentcount", "video_sharecount",
]
videos[ENGAGEMENT_COLS].describe().round(0)


### 3.2 Video-level distributions

In [ ]:
# Distribution of individual video durations.
fig, ax = plt.subplots(figsize=(11, 3.5))
sns.histplot(data=videos, x="video_duration", bins=40, ax=ax)
ax.set_title("Distribution of Individual Video Durations")
ax.set_xlabel("Duration (seconds)")
ax.set_ylabel("Number of Videos")
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of upload dates (one bar per day).
fig, ax = plt.subplots(figsize=(11, 3.5))
videos["video_date"] = videos["video_timestamp"].dt.date
date_counts = videos["video_date"].value_counts().sort_index()
ax.bar(date_counts.index, date_counts.values, edgecolor="black", width=0.9)
ax.set_title("Distribution of Video Upload Dates")
ax.set_xlabel("Date")
ax.set_ylabel("Number of Videos")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


In [ ]:
# Engagement distributions (log scale — play counts span several orders of magnitude).
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
metrics = [("video_playcount",    "Play count"),
           ("video_diggcount",    "Like (digg) count"),
           ("video_commentcount", "Comment count"),
           ("video_sharecount",   "Share count")]
for ax, (col, label) in zip(axes.flat, metrics):
    sns.histplot(videos[col].clip(lower=1), bins=40, log_scale=True, ax=ax)
    ax.set_title(label)
    ax.set_xlabel(label)
    ax.set_ylabel("Number of Videos")
plt.tight_layout()
plt.show()


### 3.3 Per-user aggregates

In [ ]:
# Build a per-user summary: video count, total duration, total plays.
per_user = (
    videos.groupby("author_username")
          .agg(video_count        = ("video_id",         "count"),
               total_duration_min = ("video_duration",   lambda s: s.sum() / 60.0),
               total_plays        = ("video_playcount",  "sum"),
               total_likes        = ("video_diggcount",  "sum"))
          .sort_values("video_count", ascending=False)
)
per_user.round(1)


In [ ]:
# Distribution of videos per user.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

per_user["video_count"].plot(kind="bar", ax=axes[0], edgecolor="black")
axes[0].set_title("Videos per User")
axes[0].set_xlabel("User")
axes[0].set_ylabel("Number of Videos")

per_user["total_duration_min"].plot(kind="bar", ax=axes[1], edgecolor="black", color="C1")
axes[1].set_title("Total Video Duration per User")
axes[1].set_xlabel("User")
axes[1].set_ylabel("Total Duration (minutes)")

for ax in axes:
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()


### 3.4 Comments analysis

Comment files use a different schema from video files — wider (29-30 cols),
with text, language, reply counts, and a `user` field stored as a
Python-literal string blob. The cells below load every comment CSV in
`sample_data/`, normalise the user blob, and ask a few simple questions:

- How many comments did each video draw?
- How long are the comments?
- What languages show up?
- Who comments the most across the sample?
- How often do comments draw replies?


In [ ]:
def _parse_user_blob(blob):
    """The `user` column is a Python literal dict-as-string. Pull the few
    fields we actually need (nickname + unique_id) and shrug off anything
    that fails to parse."""
    if not isinstance(blob, str) or not blob.startswith("{"):
        return None, None
    try:
        d = ast.literal_eval(blob)
    except Exception:
        return None, None
    return d.get("nickname"), d.get("unique_id")

comments_frames = []
for p in comment_files:
    df = pd.read_csv(p)
    # The video the comments were collected from = aweme_id (TikTok's internal name).
    df["video_id"] = df["aweme_id"].astype(str)
    df["source_file"] = p.name
    comments_frames.append(df)

comments = pd.concat(comments_frames, ignore_index=True)

# Drop duplicate rows that show up when sample_data contains a *copy* of the
# same CSV (or you accidentally glob the same file twice). cid is the
# TikTok-side primary key for a comment.
before = len(comments)
comments = comments.drop_duplicates(subset=["cid"])
dropped = before - len(comments)
if dropped:
    print(f"  (de-duplicated {dropped:,} repeated comment rows)")

# Normalise the user blob into two columns.
comments[["user_nickname", "user_unique_id"]] = comments["user"].apply(
    lambda b: pd.Series(_parse_user_blob(b))
)

# Useful derived columns.
comments["create_time"] = pd.to_datetime(comments["create_time"], unit="s", errors="coerce")
comments["text_length"] = comments["text"].fillna("").str.len()

print(f"Comments loaded : {len(comments):,}")
print(f"Unique videos   : {comments['video_id'].nunique()}")
print(f"Unique users    : {comments['user_unique_id'].nunique():,}")
print(f"Date range      : {comments['create_time'].min()} -> {comments['create_time'].max()}")


In [ ]:
# How many comments per video.
per_video = comments.groupby("video_id").size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 3.5))
per_video.plot(kind="bar", ax=ax, edgecolor="black")
ax.set_title("Comments Collected per Video")
ax.set_xlabel("video_id")
ax.set_ylabel("Number of Comments")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

per_video.head(10)


In [ ]:
# Comment length distribution (characters).
fig, ax = plt.subplots(figsize=(11, 3.5))
sns.histplot(comments["text_length"].clip(upper=500), bins=40, ax=ax)
ax.set_title("Distribution of Comment Lengths (capped at 500 chars)")
ax.set_xlabel("Comment length (characters)")
ax.set_ylabel("Number of Comments")
plt.tight_layout()
plt.show()

print("Median length :", int(comments["text_length"].median()))
print("Mean length   :", round(comments["text_length"].mean(), 1))
print("Longest       :", int(comments["text_length"].max()))


In [ ]:
# Top languages.
lang_counts = comments["comment_language"].value_counts().head(10)
fig, ax = plt.subplots(figsize=(8, 3.5))
lang_counts.plot(kind="bar", ax=ax, edgecolor="black")
ax.set_title("Top Comment Languages")
ax.set_xlabel("Language code")
ax.set_ylabel("Number of Comments")
plt.tight_layout()
plt.show()
lang_counts


In [ ]:
# Top commenters across the whole sample.
# CHANGE TOP_N TO SEE MORE OR FEWER COMMENTERS
TOP_N = 15

top_commenters = (
    comments.dropna(subset=["user_unique_id"])
            .groupby(["user_unique_id", "user_nickname"])
            .agg(comments     = ("cid",          "count"),
                 total_likes  = ("digg_count",   "sum"),
                 avg_length   = ("text_length",  "mean"))
            .sort_values("comments", ascending=False)
            .head(TOP_N)
)
top_commenters.round(1)


In [ ]:
# Reply patterns: how many comments draw at least one reply?
total          = len(comments)
with_replies   = (comments["reply_comment_total"].fillna(0) > 0).sum()
share_with_rep = 100 * with_replies / total if total else 0.0

print(f"Total comments       : {total:,}")
print(f"With at least 1 reply: {with_replies:,}  ({share_with_rep:.1f}%)")
print(f"Mean replies/comment : {comments['reply_comment_total'].fillna(0).mean():.2f}")
print(f"Max replies/comment  : {int(comments['reply_comment_total'].fillna(0).max())}")

# Distribution of reply counts (clipped at 50 so the bulk is visible).
fig, ax = plt.subplots(figsize=(11, 3.5))
sns.histplot(comments["reply_comment_total"].fillna(0).clip(upper=50),
             bins=51, ax=ax)
ax.set_title("Replies per Comment (clipped at 50)")
ax.set_xlabel("Replies")
ax.set_ylabel("Number of Comments")
plt.tight_layout()
plt.show()
